# Scenario And Storage Comparison

**Track:** PyPSA-Earth reference analysis.

**Purpose:** compare storage state of charge, delivered energy and system metrics across a selected set of solved capacity-expansion scenarios. The current scenario dictionary illustrates DEA 2030, emissions-policy, hydrogen and ammonia cases.

**Before running:** place solved networks under `pypsa-earth/results`, verify their model version and document which assumptions differ between scenarios.

**User action:** edit `SCENARIOS`, review the list of missing/skipped files, run the plots and report each comparison with its costs, emissions policy, temporal resolution and enabled technologies.

**Permitted changes:** scenario set, labels, storage carriers, metrics and visual presentation. Avoid comparisons that change multiple undocumented assumptions at once.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pypsa

# ── Scenario paths (all DEA 2030, 3 h, CCGT extendable) ──
_RESULTS = "../../pypsa-earth/results/mauritius-year-1/networks"
SCENARIOS = {
    "co2_zero_dea30": Path(f"{_RESULTS}/elec_s_140_ec_lcopt_Co2zero-3h-DEA30.nc"),
    "co2_zero_h2_dea30": Path(f"{_RESULTS}/elec_s_140_ec_lcopt_Co2zero-3h-H2-DEA30.nc"),
    "co2_zero_nh3_dea30": Path(f"{_RESULTS}/elec_s_140_ec_lcopt_Co2zero-3h-NH3-DEA30.nc"),
    "co2_limited_nh3_dea30": Path(f"{_RESULTS}/elec_s_140_ec_lcopt_Co2L-3h-NH3-DEA30.nc"),
    "co2_uncapped_nh3_dea30": Path(f"{_RESULTS}/elec_s_140_ec_lcopt_Co2nocap-3h-NH3-DEA30.nc"),
    "co2_price_dea30": Path(f"{_RESULTS}/elec_s_140_ec_lcopt_Co2nocap-Ep60-3h-DEA30.nc"),
    "co2_price_h2_dea30": Path(f"{_RESULTS}/elec_s_140_ec_lcopt_Co2nocap-Ep60-3h-H2-DEA30.nc"),
    "co2_price_nh3_dea30": Path(f"{_RESULTS}/elec_s_140_ec_lcopt_Co2nocap-Ep60-3h-NH3-DEA30.nc"),
    # ── Cost sensitivity ──
    "co2_zero_nh3_dea30_cheapbatt": Path(f"{_RESULTS}/elec_s_140_ec_lcopt_Co2zero-3h-NH3-DEA30-CheapBatt.nc"),
}

# Filter to files that exist locally
available = {k: v.resolve() for k, v in SCENARIOS.items() if v.resolve().exists()}
missing = set(SCENARIOS) - set(available)
if missing:
    print(f"Missing (skipped): {sorted(missing)}")
print(f"Loading {len(available)} networks …")

networks = {}
for label, path in available.items():
    networks[label] = pypsa.Network(str(path))
    print(f"  ✓ {label}")

In [ ]:
# ── Technology colour palette (same as 01_run_analysis) ──
TECH_COLORS = {
    "solar": "#f9d71c", "onwind": "#235ebc", "offwind-ac": "#6895dd",
    "offwind-dc": "#74c6f2", "ror": "#78d4cf",
    "OCGT": "#e05b5b", "CCGT": "#d35050", "coal": "#545454",
    "lignite": "#826a4a", "nuclear": "#ff8c00", "oil": "#2a2a2a",
    "biomass": "#baa741", "geothermal": "#ba91b1",
    "H2 electrolysis": "#ff29d2", "CCGT H2": "#c251ae",
    "H2 pipeline": "#c9aed3", "NH3 synthesis": "#46a88c",
    "CCGT NH3": "#ff69b4",
    "battery charger": "#999999", "battery discharger": "#dab813",
    "H2": "#ff29d2", "NH3": "#46a88c", "battery": "#999999",
    "load shedding": "#ff0000",
}

def tech_color(carrier):
    return TECH_COLORS.get(carrier, "#aaaaaa")

## Storage State of Charge — Comparison

One subplot per scenario showing stored energy (TWh) by carrier over the year.

In [ ]:
# ── Storage SoC subplots ──
labels = list(networks.keys())
n_plots = len(labels)
fig, axes = plt.subplots(n_plots, 1, figsize=(14, 4 * n_plots), sharex=True)
if n_plots == 1:
    axes = [axes]

# Collect all carriers across all scenarios for a unified legend
all_soc_carriers = set()
soc_data = {}
for label, net in networks.items():
    if hasattr(net, "stores_t") and "e" in net.stores_t and not net.stores_t.e.empty:
        soc = net.stores_t.e.copy()
        store_carrier = net.stores.loc[soc.columns, "carrier"]
        soc_by_carrier = soc.T.groupby(store_carrier).sum().T / 1e6  # TWh
        soc_data[label] = soc_by_carrier
        all_soc_carriers.update(soc_by_carrier.columns)
    else:
        soc_data[label] = pd.DataFrame()

sorted_carriers = sorted(all_soc_carriers)

for ax, label in zip(axes, labels):
    soc_by_carrier = soc_data[label]
    if soc_by_carrier.empty:
        ax.text(0.5, 0.5, "No store data", transform=ax.transAxes, ha="center")
    else:
        peak_order = soc_by_carrier.max().sort_values(ascending=False).index
        for carrier in peak_order:
            ax.plot(soc_by_carrier.index, soc_by_carrier[carrier],
                    label=carrier, color=tech_color(carrier), linewidth=1.2)
    ax.set_ylabel("TWh")
    ax.set_title(label, fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right", fontsize="small", frameon=True)

axes[-1].set_xlabel("Time")
fig.suptitle("Storage State of Charge by Scenario", fontsize=13, y=1.01)
fig.autofmt_xdate(rotation=30)
plt.tight_layout()
plt.show()

## Scenario Metrics Comparison

CO₂ emissions, levelised cost, total system cost, and investment share
(Generation / Transmission / Storage / Conversion) across DEA 2030 scenarios.

In [ ]:
# ── Helper functions ──

TRANSMISSION_CARRIERS = {"DC", "H2 pipeline", "NH3 pipeline"}
CONVERSION_CARRIERS = {
    "battery charger", "battery discharger",
    "H2 electrolysis", "CCGT H2",
    "NH3 synthesis", "CCGT NH3",
}

def _inv(df, cap_col, cost_col="capital_cost"):
    if df.empty or cap_col not in df.columns or cost_col not in df.columns:
        return 0.0
    opt = cap_col.replace("_nom", "_nom_opt")
    col = opt if opt in df.columns else cap_col
    return (df[col].fillna(df[cap_col]) * df[cost_col].fillna(0.0)).sum()


def compute_scenario_metrics(net):
    """Return a dict of key metrics for one solved network."""
    sw = net.snapshot_weightings.objective.reindex(net.snapshots).fillna(1.0)
    total_demand_mwh = net.loads_t.p_set.mul(sw, axis=0).sum().sum()
    objective_eur = float(getattr(net, "objective", float("nan")))

    # ── CO2 ──
    carrier_emissions = net.carriers.get("co2_emissions")
    total_co2_mt = float("nan")
    if carrier_emissions is not None and not carrier_emissions.isnull().all():
        def _comp_em(power_df, comp_df):
            if power_df is None or power_df.empty:
                return 0.0
            factors = comp_df["carrier"].map(carrier_emissions).fillna(0.0)
            return power_df.mul(sw, axis=0).mul(factors, axis=1).sum().sum()
        total_co2_mt = (
            _comp_em(net.generators_t.p, net.generators)
            + _comp_em(getattr(net.links_t, "p0", None), net.links)
            + _comp_em(getattr(net.stores_t, "p", None), net.stores)
        ) / 1e6

    # ── Investment breakdown ──
    gen_inv = _inv(net.generators, "p_nom")
    line_inv = _inv(net.lines, "s_nom")
    tx_links = net.links[net.links.carrier.isin(TRANSMISSION_CARRIERS)]
    transmission_inv = line_inv + _inv(tx_links, "p_nom")

    store_inv = _inv(net.stores, "e_nom")
    su_inv = _inv(net.storage_units, "p_nom") if not net.storage_units.empty else 0.0
    storage_inv = store_inv + su_inv

    conv_links = net.links[net.links.carrier.isin(CONVERSION_CARRIERS)]
    conversion_inv = _inv(conv_links, "p_nom")

    total_inv = gen_inv + transmission_inv + storage_inv + conversion_inv

    def _pct(v):
        return 100 * v / total_inv if total_inv > 0 else 0.0

    return {
        "total_CO2_Mt": total_co2_mt,
        "levelised_cost_EUR/MWh": objective_eur / total_demand_mwh if total_demand_mwh else float("nan"),
        "total_cost_MEUR": objective_eur / 1e6,
        "generation_%": _pct(gen_inv),
        "transmission_%": _pct(transmission_inv),
        "storage_%": _pct(storage_inv),
        "conversion_%": _pct(conversion_inv),
    }


# ── Build comparison table ──
rows = {label: compute_scenario_metrics(net) for label, net in networks.items()}
comparison_df = pd.DataFrame(rows).T
comparison_df.index.name = "scenario"

fmt = {
    "total_CO2_Mt": "{:,.1f}",
    "levelised_cost_EUR/MWh": "{:,.1f}",
    "total_cost_MEUR": "{:,.0f}",
    "generation_%": "{:.1f}",
    "transmission_%": "{:.1f}",
    "storage_%": "{:.1f}",
    "conversion_%": "{:.1f}",
}
display(comparison_df.style.format(fmt))

In [ ]:
# ── Bar charts comparing key metrics ──
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
labels = comparison_df.index.tolist()
x = np.arange(len(labels))

# Total cost
axes[0].bar(x, comparison_df["total_cost_MEUR"], color="steelblue")
axes[0].set_ylabel("MEUR")
axes[0].set_title("Total System Cost")
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, rotation=45, ha="right", fontsize=8)

# Levelised cost
axes[1].bar(x, comparison_df["levelised_cost_EUR/MWh"], color="darkorange")
axes[1].set_ylabel("EUR/MWh")
axes[1].set_title("Levelised Cost")
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels, rotation=45, ha="right", fontsize=8)

# CO2
axes[2].bar(x, comparison_df["total_CO2_Mt"], color="grey")
axes[2].set_ylabel("Mt CO₂")
axes[2].set_title("Total Operational CO₂")
axes[2].set_xticks(x)
axes[2].set_xticklabels(labels, rotation=45, ha="right", fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ── Stacked investment share bar chart ──
share_cols = ["generation_%", "transmission_%", "storage_%", "conversion_%"]
share_labels = ["Generation", "Transmission", "Storage", "Conversion"]
share_colors = ["#f9d71c", "#c9aed3", "#999999", "#ff29d2"]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(comparison_df))
bottoms = np.zeros(len(comparison_df))

for col, lbl, clr in zip(share_cols, share_labels, share_colors):
    vals = comparison_df[col].values.astype(float)
    ax.bar(x, vals, bottom=bottoms, label=lbl, color=clr)
    bottoms += vals

ax.set_ylabel("Share of Annualised Capital (%)")
ax.set_title("Investment Split by Category")
ax.set_xticks(x)
ax.set_xticklabels(comparison_df.index, rotation=45, ha="right", fontsize=8)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize="small")
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()

In [ ]:
# ── CCGT vs Nuclear economics in uncapped scenario ──
net = networks["co2_uncapped_nh3_dea30"]
sw = net.snapshot_weightings.objective.reindex(net.snapshots).fillna(1.0)

for c in ["nuclear", "CCGT", "coal", "lignite", "solar", "onwind"]:
    gens = net.generators[net.generators.carrier == c]
    if gens.empty:
        continue
    twh = net.generators_t.p[gens.index].mul(sw, axis=0).sum().sum() / 1e6
    cf = twh * 1e6 / (gens.p_nom_opt.sum() * sw.sum()) if gens.p_nom_opt.sum() > 0 else 0
    ext = gens.p_nom_extendable.any()
    added = (gens.p_nom_opt - gens.p_nom).clip(lower=0).sum() / 1e3
    print(f"{c:15s}  cap={gens.p_nom_opt.sum()/1e3:7.1f} GW  "
          f"existing={gens.p_nom.sum()/1e3:7.1f} GW  added={added:6.1f} GW  "
          f"gen={twh:7.1f} TWh  CF={cf:.1%}  "
          f"mc={gens.marginal_cost.mean():6.1f}  cc={gens.capital_cost.mean():9.0f}  "
          f"ext={ext}")

# LCOE comparison: capital_cost / (8760 * CF) + marginal_cost
print("\n--- Indicative LCOE at observed capacity factors ---")
for c in ["nuclear", "CCGT", "coal", "lignite"]:
    gens = net.generators[net.generators.carrier == c]
    if gens.empty:
        continue
    twh = net.generators_t.p[gens.index].mul(sw, axis=0).sum().sum() / 1e6
    cf = twh * 1e6 / (gens.p_nom_opt.sum() * sw.sum()) if gens.p_nom_opt.sum() > 0 else 0
    cc = gens.capital_cost.mean()
    mc = gens.marginal_cost.mean()
    lcoe = cc / (8760 * cf) + mc if cf > 0 else float("inf")
    print(f"  {c:15s}  CC={cc:9.0f} EUR/MW/yr  MC={mc:6.1f} EUR/MWh  CF={cf:.1%}  → LCOE≈{lcoe:.1f} EUR/MWh")


## Storage Delivered Energy by Carrier (TWh)

Summary of energy delivered (discharged) from each storage carrier across all
scenarios — mirrors the cycling summary table from `01_run_analysis`.

In [ ]:
# ── Delivered TWh from each storage carrier, per scenario ──
delivered_rows = {}

for label, n in networks.items():
    if not (hasattr(n, "stores_t") and "p" in n.stores_t and not n.stores_t.p.empty):
        continue
    if not (hasattr(n, "stores_t") and "e" in n.stores_t and not n.stores_t.e.empty):
        continue

    p = n.stores_t.p.copy()  # MW per store per snapshot
    store_carrier = n.stores.loc[p.columns, "carrier"]

    # Snapshot weights
    if "generators" in n.snapshot_weightings.columns:
        weights = n.snapshot_weightings.loc[p.index, "generators"]
    else:
        weights = pd.Series(1.0, index=p.index)

    # Discharge: p < 0 means energy delivered to the bus
    discharge_energy = p.clip(upper=0).abs().mul(weights, axis=0).sum()  # MWh per store
    delivered_by_carrier = discharge_energy.groupby(store_carrier).sum() / 1e6  # TWh

    delivered_rows[label] = delivered_by_carrier

delivered_df = pd.DataFrame(delivered_rows).T.fillna(0.0)
delivered_df.index.name = "scenario"

# Sort columns by total delivered (largest first)
delivered_df = delivered_df[delivered_df.sum().sort_values(ascending=False).index]

# Add a total column
delivered_df["TOTAL"] = delivered_df.sum(axis=1)

display(delivered_df.style.format("{:,.2f}").set_caption("Delivered Energy by Storage Carrier (TWh)"))